## 1. Environment Setup & Dependencies
Install the required libraries on Google Colab.

In [ ]:
# Install dependencies
!pip install -q faiss-gpu sentence-transformers whoosh transformers accelerate torch

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# Configure paths
DRIVE_BASE = Path('/content/drive/MyDrive/legal_rag')
VECTOR_DIR = DRIVE_BASE / 'vector_store'
CACHE_DIR = DRIVE_BASE / 'cache'
MODEL_DIR = DRIVE_BASE / 'models' / 'qwen_0.5b'

# Sanity check paths
print('Drive base directory exists:', DRIVE_BASE.exists())
print('Vector store directory exists:', VECTOR_DIR.exists())
print('Cache directory exists:', CACHE_DIR.exists())
print('Model directory exists:', MODEL_DIR.exists())

Mounted at /content/drive
Drive base directory exists: True
Vector store directory exists: True
Cache directory exists: True
Model directory exists: False


## 3. Core RAG Retrieval & Influence Scoring Pipeline
This cell implements the memory-efficient streaming retrieval, FAISS search, Whoosh BM25 search, hybrid normalization, and citation influence scoring.

In [ ]:
import json
import re
import numpy as np
import faiss
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from whoosh import index
from whoosh.qparser import QueryParser, OrGroup

In [ ]:
import json
import re
import numpy as np
import faiss
import torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
from whoosh import index
from whoosh.qparser import QueryParser, OrGroup

# File paths
FAISS_FILE = VECTOR_DIR / 'legal_cases.index'
METADATA_FILE = VECTOR_DIR / 'metadata.jsonl'
OFFSETS_FILE = VECTOR_DIR / 'metadata_offsets.bin'
JURISDICTIONS_FILE = VECTOR_DIR / 'jurisdiction_ranges.json'
BM25_DIR = VECTOR_DIR / 'bm25_index'
CITATION_COUNTS_PATH = CACHE_DIR / 'citation_counts.json'

# Scoring hyperparameters
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
FAISS_TOP_K = 50
BM25_TOP_K = 50
FINAL_TOP_K = 10
BETA = 0.4
GAMMA = 0.7

def normalize_citation(citation):
    if not citation:
        return None
    citation = str(citation).upper().strip()
    citation = re.sub(r'[^A-Z0-9]+', ' ', citation)
    return re.sub(r'\s+', ' ', citation).strip()

def jurisdiction_matches(document_jurisdiction, jurisdiction):
    if jurisdiction is None or jurisdiction == 'All':
        return True
    if jurisdiction in ['USA', 'USA - South Carolina']:
        return str(document_jurisdiction).startswith('USA')
    if jurisdiction == 'Australia':
        return str(document_jurisdiction) == 'Australia'
    if jurisdiction == 'Singapore':
        return str(document_jurisdiction) == 'Singapore'
    return str(document_jurisdiction).lower() == str(jurisdiction).lower()

def load_documents_for_positions(metadata_file, positions):
    position_set = set(int(p) for p in positions if p >= 0)
    documents = {}
    if not position_set:
        return documents

    # O(1) random-access seek using precomputed offsets
    if OFFSETS_FILE.exists():
        try:
            offsets = np.memmap(str(OFFSETS_FILE), dtype=np.int64, mode='r')
            with open(metadata_file, 'r', encoding='utf-8') as f:
                for pos in position_set:
                    if 0 <= pos < len(offsets):
                        f.seek(int(offsets[pos]))
                        line = f.readline()
                        if line:
                            documents[pos] = json.loads(line)
            return documents
        except Exception as e:
            print(f'Offset seek fallback: {e}')

    # Fallback to streaming line-by-line
    with open(metadata_file, 'r', encoding='utf-8') as f:
        for position, line in enumerate(f):
            if position not in position_set:
                continue
            documents[position] = json.loads(line)
            if len(documents) == len(position_set):
                break
    return documents

def load_documents_for_chunks(metadata_file, chunk_ids):
    chunk_ids = set(str(cid) for cid in chunk_ids if cid)
    documents = {}
    if not chunk_ids:
        return documents

    with open(metadata_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            doc = json.loads(line)
            cid = str(doc.get('chunk_id', ''))
            if cid in chunk_ids:
                documents[cid] = doc
                if len(documents) == len(chunk_ids):
                    break
    return documents

def get_jurisdiction_range(jurisdiction):
    if jurisdiction is None or jurisdiction == 'All':
        return None
    if JURISDICTIONS_FILE.exists():
        try:
            with open(JURISDICTIONS_FILE, 'r', encoding='utf-8') as f:
                ranges = json.load(f)
            for jur_key, span in ranges.items():
                if jurisdiction_matches(jur_key, jurisdiction):
                    return span
        except Exception:
            pass
    return None

def faiss_search(query, model, faiss_index, metadata_file, jurisdiction=None):
    query_embedding = model.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    jur_range = get_jurisdiction_range(jurisdiction)

    if jur_range is not None and hasattr(faiss, 'SearchParameters') and hasattr(faiss, 'IDSelectorRange'):
        start, end = jur_range
        sel = faiss.IDSelectorRange(int(start), int(end))
        params = faiss.SearchParameters(sel=sel)
        scores, positions = faiss_index.search(query_embedding, FAISS_TOP_K, params=params)
        results = [{'position': int(pos), 'score': float(score)} for score, pos in zip(scores[0], positions[0]) if pos != -1]
        return results

    fetch_k = FAISS_TOP_K if (jurisdiction is None or jurisdiction == 'All') else min(FAISS_TOP_K * 6, faiss_index.ntotal)
    scores, positions = faiss_index.search(query_embedding, fetch_k)

    raw_positions = [int(p) for p in positions[0] if p != -1]
    if jurisdiction is None or jurisdiction == 'All':
        return [{'position': int(pos), 'score': float(score)} for score, pos in zip(scores[0], positions[0]) if pos != -1][:FAISS_TOP_K]

    docs = load_documents_for_positions(metadata_file, raw_positions)
    results = []
    for score, pos in zip(scores[0], positions[0]):
        if pos == -1:
            continue
        doc = docs.get(pos)
        if doc and jurisdiction_matches(doc.get('jurisdiction', ''), jurisdiction):
            results.append({'position': int(pos), 'score': float(score)})
            if len(results) >= FAISS_TOP_K:
                break
    return results

def bm25_search(query, bm25_index, metadata_file, jurisdiction=None):
    parser = QueryParser('text', schema=bm25_index.schema, group=OrGroup)
    try:
        parsed_query = parser.parse(query)
    except Exception:
        clean_q = re.sub(r'[^A-Za-z0-9\s]', ' ', query)
        parsed_query = parser.parse(clean_q)

    fetch_limit = BM25_TOP_K if (jurisdiction is None or jurisdiction == 'All') else BM25_TOP_K * 4
    results = []
    with bm25_index.searcher() as searcher:
        matches = searcher.search(parsed_query, limit=fetch_limit)
        for result in matches:
            results.append({'chunk_id': str(result['chunk_id']), 'score': float(result.score)})

    if jurisdiction is None or jurisdiction == 'All':
        return results[:BM25_TOP_K]

    chunk_ids = [r['chunk_id'] for r in results]
    documents = load_documents_for_chunks(metadata_file, chunk_ids)
    filtered = []
    for r in results:
        doc = documents.get(r['chunk_id'])
        if doc and jurisdiction_matches(doc.get('jurisdiction', ''), jurisdiction):
            filtered.append(r)
            if len(filtered) >= BM25_TOP_K:
                break
    return filtered

def create_candidates(faiss_results, bm25_results, metadata_file):
    candidates = {}
    faiss_positions = [r['position'] for r in faiss_results]
    faiss_documents = load_documents_for_positions(metadata_file, faiss_positions)
    bm25_chunk_ids = [r['chunk_id'] for r in bm25_results]
    bm25_documents = load_documents_for_chunks(metadata_file, bm25_chunk_ids)

    def make_candidate(document):
        return {
            'case_id': str(document.get('case_id')),
            'case_name': document.get('case_name'),
            'jurisdiction': document.get('jurisdiction'),
            'court': document.get('court'),
            'citation': document.get('citation'),
            'decision_date': document.get('decision_date'),
            'outcome': document.get('outcome'),
            'acts': document.get('acts', []),
            'sections': document.get('sections', []),
            'chunks': [],
            'best_faiss_score': 0.0,
            'best_bm25_score': 0.0
        }

    for r in faiss_results:
        doc = faiss_documents.get(r['position'])
        if not doc:
            continue
        cid = str(doc.get('case_id'))
        if cid not in candidates:
            candidates[cid] = make_candidate(doc)
        candidates[cid]['best_faiss_score'] = max(candidates[cid]['best_faiss_score'], r['score'])
        candidates[cid]['chunks'].append({'chunk_id': str(doc.get('chunk_id')), 'source': 'FAISS', 'score': r['score'], 'text': doc.get('text', '')})

    for r in bm25_results:
        doc = bm25_documents.get(r['chunk_id'])
        if not doc:
            continue
        cid = str(doc.get('case_id'))
        if cid not in candidates:
            candidates[cid] = make_candidate(doc)
        candidates[cid]['best_bm25_score'] = max(candidates[cid]['best_bm25_score'], r['score'])
        candidates[cid]['chunks'].append({'chunk_id': r['chunk_id'], 'source': 'BM25', 'score': r['score'], 'text': doc.get('text', '')})

    return list(candidates.values())

def min_max_normalize(val, v_min, v_max):
    return (val - v_min) / (v_max - v_min) if v_max > v_min else 0.0

def normalize_retrieval_scores(candidates):
    if not candidates:
        return candidates
    f_scores = [c['best_faiss_score'] for c in candidates]
    b_scores = [c['best_bm25_score'] for c in candidates]
    f_min, f_max = min(f_scores), max(f_scores)
    b_min, b_max = min(b_scores), max(b_scores)

    for c in candidates:
        c['normalized_faiss'] = min_max_normalize(c['best_faiss_score'], f_min, f_max)
        c['normalized_bm25'] = min_max_normalize(c['best_bm25_score'], b_min, b_max)
        c['hybrid_score'] = 0.5 * c['normalized_faiss'] + 0.5 * c['normalized_bm25']
    return candidates

def calculate_influence_scores(candidates, citation_counts):
    if not candidates:
        return candidates
    c_vals = []
    for c in candidates:
        cit = c.get('citation')
        cid = c.get('case_id')
        norm_cit = normalize_citation(cit)
        cnt = citation_counts.get(cit, 0) or citation_counts.get(norm_cit, 0) or citation_counts.get(cid, 0)
        c['citation_count'] = cnt
        c_vals.append(cnt)

    maximum = max(c_vals, default=0)
    for c in candidates:
        c['normalized_citation_count'] = (c['citation_count'] / maximum) if maximum > 0 else 0.0
        c['influence_score'] = BETA * c['normalized_citation_count'] + (1.0 - BETA) * c['hybrid_score']
    return candidates

def calculate_final_scores(candidates):
    for c in candidates:
        c['final_score'] = GAMMA * c['hybrid_score'] + (1.0 - GAMMA) * c['influence_score']
    candidates.sort(key=lambda x: x['final_score'], reverse=True)
    return candidates

print('Retrieval & Scoring functions loaded successfully.')

Retrieval & Scoring functions loaded successfully.


## 4. Context Builder & LLM Generator (Qwen)
Loads `Qwen/Qwen2.5-0.5B-Instruct` with GPU acceleration and defines the grounded generation prompt.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Check for local weights or fallback to HuggingFace
TARGET_MODEL = MODEL_DIR if MODEL_DIR.exists() else 'Qwen/Qwen2.5-0.5B-Instruct'

print(f'Loading Qwen from: {TARGET_MODEL}...')
tokenizer = AutoTokenizer.from_pretrained(str(TARGET_MODEL))
llm_model = AutoModelForCausalLM.from_pretrained(
    str(TARGET_MODEL),
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto' if torch.cuda.is_available() else None
)

SYSTEM_PROMPT = """
You are a legal research assistant.

Answer the user's question using only the legal authorities and passages supplied in the context.

Do not invent cases, citations, statutes, sections, facts, holdings, or legal propositions.

Distinguish between what an authority explicitly supports and what is merely an inference.

If the retrieved authorities do not provide enough information to answer the question reliably, say so.

Do not treat the user's assumptions as established legal facts.

Provide a concise explanation of the relevant law.

Cite the supplied authorities when making material legal claims.

Do not create citations that are not present in the provided context.

If the retrieved context contains no relevant authorities, do not generate case names, citations, statutes, facts, or legal conclusions.

Instead, state that no relevant authorities were retrieved from the supplied corpus.

Never use placeholders such as [Case Name], [Court Name], [Date], etc.
"""

def build_context(candidates, max_cases=5, max_chunks_per_case=3):
    context_cases = []
    for rank, c in enumerate(candidates[:max_cases], start=1):
        sorted_chunks = sorted(c['chunks'], key=lambda x: x.get('score', 0.0), reverse=True)[:max_chunks_per_case]
        passages = [chunk['text'].strip() for chunk in sorted_chunks if chunk.get('text')]
        context_cases.append({
            'rank': rank,
            'case_name': c.get('case_name'),
            'citation': c.get('citation'),
            'jurisdiction': c.get('jurisdiction'),
            'court': c.get('court'),
            'passages': passages
        })
    return context_cases

def format_context(context_cases):
    lines = []
    for case in context_cases:
        lines.append(f"Authority {case['rank']}: {case['case_name']} ({case['citation']}) - {case['jurisdiction']}")
        lines.append(f"Court: {case['court']}")
        for p_idx, p in enumerate(case['passages'], start=1):
            lines.append(f"Passage {p_idx}:\n{p}")
        lines.append('-' * 40)
    return '\n'.join(lines)

def generate_answer(query, formatted_context, tokenizer, model):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT.strip()},
        {'role': 'user', 'content': f"USER QUERY:\n\n{query}\n\nRETRIEVED LEGAL AUTHORITIES:\n\n{formatted_context}\n\nAnswer the query using only the retrieved authorities."}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=700, do_sample=False)
    tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(tokens, skip_special_tokens=True).strip()

print('Context Builder & Qwen LLM loaded.')

Loading Qwen from: Qwen/Qwen2.5-0.5B-Instruct...


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Context Builder & Qwen LLM loaded.


## 5. Main Retrieval & Answering Function

In [ ]:
# Load retrieval components once
print('Loading FAISS index...')
faiss_index = faiss.read_index(str(FAISS_FILE))
print(f'FAISS vectors: {faiss_index.ntotal:,}')

print('Loading BM25 index...')
bm25_idx = index.open_dir(str(BM25_DIR))
print(f'BM25 documents: {bm25_idx.doc_count():,}')

print('Loading embedding model on GPU...')
embed_model = SentenceTransformer(MODEL_NAME, device='cuda' if torch.cuda.is_available() else 'cpu')

print('Loading citation counts...')
with open(CITATION_COUNTS_PATH, 'r', encoding='utf-8') as f:
    citation_counts = json.load(f)
print(f'Loaded {len(citation_counts):,} citation entries.')

def legal_rag_pipeline(query: str, jurisdiction: str = 'All'):
    print('=' * 80)
    print(f'LEGAL QUERY: {query}')
    print(f'JURISDICTION: {jurisdiction}')
    print('=' * 80)

    # Step 1: Hybrid Retrieval
    f_res = faiss_search(query, embed_model, faiss_index, METADATA_FILE, jurisdiction)
    b_res = bm25_search(query, bm25_idx, METADATA_FILE, jurisdiction)
    candidates = create_candidates(f_res, b_res, METADATA_FILE)
    candidates = normalize_retrieval_scores(candidates)

    # Step 2: Influence & Final Scoring
    candidates = calculate_influence_scores(candidates, citation_counts)
    candidates = calculate_final_scores(candidates)

    print(f'\nFAISS Hits: {len(f_res)} | BM25 Hits: {len(b_res)} | Deduplicated Cases: {len(candidates)}')
    if not candidates:
        print('No relevant legal authorities found.')
        return

    print('\n' + '=' * 80)
    print('TOP RETRIEVED PRECEDENTS')
    print('=' * 80)
    for rank, c in enumerate(candidates[:FINAL_TOP_K], start=1):
        print(f"{rank}. [{c['jurisdiction']}] {c['case_name']} ({c['citation']})")
        print(f"   Final Score: {c['final_score']:.4f} | Hybrid: {c['hybrid_score']:.4f} | Influence: {c['influence_score']:.4f}")
        print(f"   Citation Count: {c['citation_count']} (Norm: {c['normalized_citation_count']:.4f}) | Court: {c['court']}")

    # Step 3: Context Building & Answer Generation
    ctx = build_context(candidates)
    fmt_ctx = format_context(ctx)

    print('\n' + '=' * 80)
    print('GROUNDED LEGAL ANSWER')
    print('=' * 80)
    answer = generate_answer(query, fmt_ctx, tokenizer, llm_model)
    print(answer)

    print('\n' + '=' * 80)
    print('RETRIEVED AUTHORITIES CITED')
    print('=' * 80)
    for case in ctx:
        print(f"{case['rank']}. {case['case_name']} ({case['citation']}) - {case['jurisdiction']}")

Loading FAISS index...
FAISS vectors: 689,259
Loading BM25 index...
BM25 documents: 689,259
Loading embedding model on GPU...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loading citation counts...
Loaded 200,479 citation entries.


## 6. Sample Queries Across Jurisdictions
Test retrieval and grounded answering across Australia, Singapore, USA - South Carolina, and All.

In [ ]:
# Query 1: Australia
legal_rag_pipeline(
    query="How did the High Court of Australia approach the recognition of native title and the rejection of terra nullius?",
    jurisdiction="Australia"
)

LEGAL QUERY: How did the High Court of Australia approach the recognition of native title and the rejection of terra nullius?
JURISDICTION: Australia

FAISS Hits: 50 | BM25 Hits: 32 | Deduplicated Cases: 41

TOP RETRIEVED PRECEDENTS
1. [Australia] Griffiths v Minister for Lands, Planning and Environment ([2008] HCA 20)
   Final Score: 0.8796 | Hybrid: 0.9995 | Influence: 0.5997
   Citation Count: 0 (Norm: 0.0000) | Court: High Court of Australia
2. [Australia] Western Australia v Augustine | Commonwealth of Australia v Augustine | Commonwealth of Australia v Manado ([2020] HCA 9)
   Final Score: 0.8306 | Hybrid: 0.9438 | Influence: 0.5663
   Citation Count: 0 (Norm: 0.0000) | Court: High Court of Australia
3. [Australia] Members of the Yorta Yorta Aboriginal Community v Victoria ([2002] HCA 58)
   Final Score: 0.8303 | Hybrid: 0.9435 | Influence: 0.5661
   Citation Count: 0 (Norm: 0.0000) | Court: High Court of Australia
4. [Australia] Commonwealth of Australia v Yunupingu ([2025] HCA 

In [ ]:
# Query 2: Singapore
legal_rag_pipeline(
    query="breach of contract damages and fiduciary duty in commercial transactions",
    jurisdiction="Singapore"
)

LEGAL QUERY: breach of contract damages and fiduciary duty in commercial transactions
JURISDICTION: Singapore

FAISS Hits: 50 | BM25 Hits: 50 | Deduplicated Cases: 49

TOP RETRIEVED PRECEDENTS
1. [Singapore] Goh Chan Peng and others v Beyonics Technology Ltd and another and another appeal [2017] SGCA 40 ([2017] SGCA 40)
   Final Score: 0.8309 | Hybrid: 0.9442 | Influence: 0.5665
   Citation Count: 0 (Norm: 0.0000) | Court: Singapore Court of Appeal
2. [Singapore] Viking Airtech Pte Ltd v Foo Teow Keng and Another [2009] SGHC 169 ([2009] SGHC 169)
   Final Score: 0.8102 | Hybrid: 0.9207 | Influence: 0.5524
   Citation Count: 0 (Norm: 0.0000) | Court: Singapore High Court
3. [Singapore] Nordic International Ltd v Morten Innhaug [2017] SGHC 1 ([2017] SGHC 1)
   Final Score: 0.8001 | Hybrid: 0.9092 | Influence: 0.5455
   Citation Count: 0 (Norm: 0.0000) | Court: Singapore High Court
4. [Singapore] Friis and Another v Casetech Trading Pte Ltd and Others [2000] SGCA 35 ([2000] SGCA 35)
   Fi

In [ ]:
# Query 3: USA - South Carolina
legal_rag_pipeline(
    query="negligence duty of care personal injury damages",
    jurisdiction="USA - South Carolina"
)

LEGAL QUERY: negligence duty of care personal injury damages
JURISDICTION: USA - South Carolina

FAISS Hits: 31 | BM25 Hits: 42 | Deduplicated Cases: 49

TOP RETRIEVED PRECEDENTS
1. [USA - South Carolina] SULLIVAN v. SOUTHERN RY. (74 S.C. 377)
   Final Score: 0.8305 | Hybrid: 0.9438 | Influence: 0.5663
   Citation Count: 0 (Norm: 0.0000) | Court: Supreme Court of South Carolina
2. [USA - South Carolina] WHITE v. RAILROAD COMPANY (30 S.C. 218)
   Final Score: 0.8267 | Hybrid: 0.9395 | Influence: 0.5637
   Citation Count: 0 (Norm: 0.0000) | Court: Supreme Court of South Carolina
3. [USA - South Carolina] JAMES v. FOUNTAIN INN MFG. CO. (80 S.C. 232)
   Final Score: 0.8217 | Hybrid: 0.9337 | Influence: 0.5602
   Citation Count: 0 (Norm: 0.0000) | Court: Supreme Court of South Carolina
4. [USA - South Carolina] BRASINGTON v. SOUTH BOUND R. R. CO. (62 S.C. 325)
   Final Score: 0.8115 | Hybrid: 0.9221 | Influence: 0.5533
   Citation Count: 0 (Norm: 0.0000) | Court: Supreme Court of South Caro

In [ ]:
# Query 4: All Jurisdictions
legal_rag_pipeline(
    query="criminal liability appeal against conviction and evidence admissibility",
    jurisdiction="All"
)

LEGAL QUERY: criminal liability appeal against conviction and evidence admissibility
JURISDICTION: All

FAISS Hits: 50 | BM25 Hits: 50 | Deduplicated Cases: 48

TOP RETRIEVED PRECEDENTS
1. [Singapore] Chua Boon Chye v Public Prosecutor [2015] SGCA 31 ([2015] SGCA 31)
   Final Score: 0.8416 | Hybrid: 0.9563 | Influence: 0.5738
   Citation Count: 0 (Norm: 0.0000) | Court: Singapore Court of Appeal
2. [Singapore] Ong Bee Nah v Won Siew Wan (Yong Tian Choy, Third Party) [2005] SGHC 52 ([2005] SGHC 52)
   Final Score: 0.8275 | Hybrid: 0.9403 | Influence: 0.5642
   Citation Count: 0 (Norm: 0.0000) | Court: Singapore High Court
3. [Singapore] Muhammad bin Kadar and another v Public Prosecutor [2011] SGCA 32 ([2011] SGCA 32)
   Final Score: 0.8170 | Hybrid: 0.9284 | Influence: 0.5570
   Citation Count: 0 (Norm: 0.0000) | Court: Singapore Court of Appeal
4. [Singapore] Mohammad Farid bin Batra v Public Prosecutor and another appeal and other matter s [2020] SGCA 19 ([2020] SGCA 19)
   Final Sco

In [ ]:
import json

METADATA_FILE="/content/drive/MyDrive/legal_rag/vector_store/metadata.jsonl"

found=False

with open(METADATA_FILE,"r",encoding="utf-8") as f:
    for line in f:
        data=json.loads(line)
        if data.get("jurisdiction")=="Australia":
            found=True
            print("Australia found!")
            print(data)
            break

print("Australia present:",found)

Australia present: False
